In [6]:
import FinanceDataReader as fdr
import pandas as pd
import os

class StockAnalyzer:
    def __init__(self, code, name):
        self.code = code
        self.name = name
        self.df = None

    def fetch_data(self, start_date):
        print(f"[{self.name}] 데이터를 가져오는 중...")
        self.df = fdr.DataReader(self.code, start_date)
        print(f"[{self.name}] 수집 완료! (데이터 수: {len(self.df)}개)")

    def apply_strategy(self):
        if self.df is None: return

        self.df['MA5'] = self.df['Close'].rolling(window=5).mean()
        self.df['MA20'] = self.df['Close'].rolling(window=20).mean()

        self.df['Signal'] = (self.df['MA5'] > self.df['MA20']).astype(int)

        self.df['Daily_Return'] = self.df['Close'].pct_change()
        self.df['Strategy_Return'] = self.df['Daily_Return'] * self.df['Signal'].shift(1)

        self.df['Cum_Return'] = (1+self.df['Strategy_Return'].fillna(0)).cumprod()

        print(f"[{self.name}] 전략 적용 및 수익률 계산 완료!")

    def get_avg_volume(self):
        return self.df['Volume'].mean()

    def check_target(self, target_price):
        hit_days = self.df[self.df['High'] >= target_price]
        print(f"--- {self.name} 분석 결과 ---")
        print(f"목표가 {target_price:,.0f}원을 넘긴 날: {len(hit_days)}일")

cosmo_chemical = StockAnalyzer('005420', '코스모화학')
cosmo_chemical.fetch_data('2025-01-01')
cosmo_chemical.apply_strategy()
cosmo_chemical.check_target(20000)

print(f"\n[최근 전략 신호 (1:매수, 0:매도)]")
print(cosmo_chemical.df[['Close', 'MA5', 'MA20', 'Signal', 'Cum_Return']].tail())

with pd.ExcelWriter('코스모화학_전략분석.xlsx') as writer:
    cosmo_chemical.df.to_excel(writer, sheet_name='전략결과')

print("\n" + "="*30)
print(f"현재 위치: {os.getcwd()}")
print("엑셀 저장 및 분석이 모두 완료되었습니다. 이제 GitHub에 업데이트 하세요!")

[코스모화학] 데이터를 가져오는 중...
[코스모화학] 수집 완료! (데이터 수: 281개)
[코스모화학] 전략 적용 및 수익률 계산 완료!
--- 코스모화학 분석 결과 ---
목표가 20,000원을 넘긴 날: 30일

[최근 전략 신호 (1:매수, 0:매도)]
            Close      MA5     MA20  Signal  Cum_Return
Date                                                   
2026-02-24  17660  17242.0  16702.0       1    0.901539
2026-02-25  17740  17416.0  16802.5       1    0.905623
2026-02-26  17190  17388.0  16851.5       1    0.877546
2026-02-27  17070  17392.0  16905.0       1    0.871420
2026-03-03  15500  17032.0  16815.5       1    0.791271

현재 위치: C:\WINDOWS\system32
엑셀 저장 및 분석이 모두 완료되었습니다. 이제 GitHub에 업데이트 하세요!
